In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
WORK = '/kaggle/working'

# decision settings, fixed before looking at any result in this notebook
TH = 0.003                      # a change must improve the mean fresh-fold proxy by at least this much
FRESH, SEL = ['A', 'B', 'C'], ['g250', 'g120']
ALL_S = FRESH + SEL

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
folds = pd.read_csv(find('folds.csv'))
fg = pd.read_csv(find('folds_group.csv'))
folds = folds.merge(fg[['uid', 'g120', 'g250']], on='uid', how='left')
assert folds[['g120', 'g250']].isna().sum().sum() == 0
te_uids = ['test/' + f for f in test.filename]
n_tr = len(folds)
AM = pd.read_csv(find('audio_meta.csv')).set_index('uid')
dur_te = AM.duration.reindex(te_uids).values
te_grp = np.where(dur_te < 40, 'short', np.where(dur_te < 52, 'dur_45s', 'dur_60s'))

y = folds.label.values
w = folds.w_test.values
nz = (folds.is_zero == 0).values
grp = folds.dur_group.values
idx_nz = np.where(nz)[0]

# ---------- metrics ----------
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def pear(a, b): return float(np.corrcoef(a, b)[0, 1])
def wrmse(a, b, ww): return float(np.sqrt(np.sum(ww * (a - b) ** 2) / np.sum(ww)))
def wpear(a, b, ww):
    ma, mb = np.average(a, weights=ww), np.average(b, weights=ww)
    return float(np.sum(ww * (a - ma) * (b - mb)) /
                 np.sqrt(np.sum(ww * (a - ma) ** 2) * np.sum(ww * (b - mb) ** 2)))
def score(oof):
    rm, pe = wrmse(y[nz], oof[nz], w[nz]), wpear(y[nz], oof[nz], w[nz])
    out = dict(proxy=(rm + 1 - pe) / 2, rmse_w=rm, pear_w=pe)
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        out['pear_' + g[4:6]] = pear(y[m], oof[m])
    return out

# ---------- features ----------
def load_emb(emb_file, uid_file):
    E = np.load(find(emb_file), mmap_mode='r')
    ids = pd.read_csv(find(uid_file)).uid.tolist()
    ix = {u: i for i, u in enumerate(ids)}
    return E, np.array([ix[u] for u in folds.uid]), np.array([ix[u] for u in te_uids])

EW, w_tr, w_te = load_emb('wavlm_large_mean.npy', 'wavlm_uids.csv')
ES = np.load(find('wavlm_large_std.npy'), mmap_mode='r')
EH, h_tr, h_te = load_emb('whisper_enc_mean.npy', 'whisper_enc_uids.csv')
E7, s_tr, s_te = load_emb('llm_rubric_meanpool.npy', 'llm_views_uids.csv')
E14, q_tr, q_te = load_emb('llm14_meanpool.npy', 'llm14_uids.csv')
j7, j14 = [6, 9, 12, 16, 20, 24].index(16), [10, 15, 21, 27, 34, 41].index(21)

AW = (np.asarray(EW[w_tr], dtype=np.float32), np.asarray(EW[w_te], dtype=np.float32))
AH = (np.asarray(EH[h_tr], dtype=np.float32), np.asarray(EH[h_te], dtype=np.float32))
FEAT = {'W': (AW[0][:, 21], AW[1][:, 21]),
        'H': (AH[0][:, 30], AH[1][:, 30]),
        'M7': (np.asarray(E7[s_tr, j7], dtype=np.float32), np.asarray(E7[s_te, j7], dtype=np.float32)),
        'M14': (np.asarray(E14[q_tr, j14], dtype=np.float32), np.asarray(E14[q_te, j14], dtype=np.float32))}

TF = pd.read_csv(find('text_features.csv'), index_col='uid')
LS = pd.read_csv(find('llm_scores.csv')).set_index('uid')
Q = pd.read_csv(find('llm14_scores.csv')).set_index('uid')
llm_cols = ['e', 'ent'] + [f'p{k}' for k in range(1, 6)] + [f'l{k}' for k in range(1, 6)]
q_cols = ['e', 'ent', 'p1', 'p2', 'p3', 'p4', 'p5']
def text_mat(uids):
    return np.hstack([TF.loc[uids].values, LS.loc[uids, llm_cols].values,
                      Q.loc[uids, q_cols].values]).astype(np.float32)
FEAT['TL14'] = (text_mat(list(folds.uid)), text_mat(te_uids))
TEXT_TAGS = {'TL14'}

enc_ids = pd.read_csv(find('enc_uids.csv')).uid.tolist()
assert enc_ids[:n_tr] == folds.uid.tolist() and enc_ids[n_tr:] == te_uids, 'enc uid order mismatch'
ENC = {}
for t in ['hubert', 'w2v2']:
    M = np.load(find(f'{t}_mean.npy')).astype(np.float32)
    S = np.load(find(f'{t}_std.npy')).astype(np.float32)
    assert not np.isnan(M).any() and not np.isnan(S).any(), t
    ENC[t] = ((M[:n_tr], S[:n_tr]), (M[n_tr:], S[n_tr:]))
print({k: v[0].shape for k, v in FEAT.items()}, '| encoders:', {t: v[0][0].shape for t, v in ENC.items()})

# ---------- models ----------
class Winsor(BaseEstimator, TransformerMixin):
    def __init__(self, lo=1, hi=99):
        self.lo, self.hi = lo, hi
    def fit(self, X, y=None):
        self.lo_ = np.percentile(X, self.lo, axis=0)
        self.hi_ = np.percentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(X, self.lo_, self.hi_)

def ridge_text():
    return make_pipeline(SimpleImputer(strategy='median'), Winsor(), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-1, 4, 12)))
def ridge_hid():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 6, 11)))
def ridge_wavlm():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 9)))
def svr_head():
    return make_pipeline(StandardScaler(), PCA(n_components=128, random_state=0),
                         SVR(C=3.0, epsilon=0.2, gamma='scale'))
MAKER = {'W': ridge_wavlm, 'TL14': ridge_text}

def fit_model(maker, X, yy, sw=None):
    m = maker()
    if sw is None:
        return m.fit(X, yy)
    return m.fit(X, yy, **{f'{m.steps[-1][0]}__sample_weight': sw})

def oof_for(X, fold, maker, sw=None):
    oof = np.full(n_tr, np.nan)
    for f in range(5):
        va = fold == f
        tr = (~va) & nz
        m = fit_model(maker, X[tr], y[tr], None if sw is None else sw[tr])
        oof[va & nz] = np.clip(m.predict(X[va & nz]), 1, 5)
    return oof

# ---------- folds (same code and seeds as notebook 15) ----------
ybin = np.digitize(y, [2.25, 3.25, 4.25])
strat = ybin * 2 + (grp == 'dur_60s').astype(int)
def make_folds(layers, k, seed):
    mean_ = np.asarray(EW[w_tr][:, layers, :], dtype=np.float32).reshape(n_tr, -1)
    std_ = np.asarray(ES[w_tr][:, layers, :], dtype=np.float32).reshape(n_tr, -1)
    G = np.hstack([mean_, std_])
    sc = StandardScaler().fit(G[nz])
    Z = PCA(n_components=50, random_state=seed).fit(sc.transform(G[nz])).transform(sc.transform(G))
    cl = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average').fit_predict(Z[nz])
    fold = np.random.RandomState(seed).randint(0, 5, n_tr)
    sg = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for f, (_, va) in enumerate(sg.split(idx_nz, strat[nz], groups=cl)):
        fold[idx_nz[va]] = f
    return fold

SCHEMES = {'A': ((0, 2, 4), 180, 7), 'B': ((1, 3, 5), 90, 11), 'C': ((2, 4, 6), 250, 23)}
FOLDS = {n: make_folds(*a) for n, a in SCHEMES.items()}
FOLDS['g250'], FOLDS['g120'] = folds.g250.values, folds.g120.values
print('fold sizes (non-zero):', {s: np.bincount(FOLDS[s][nz], minlength=5).tolist() for s in ALL_S})

# ---------- cached component OOFs ----------
SWS = {'none': None}
CACHE = {}
def comp_oof(tag, s, sw='none'):
    key = (tag, s, sw)
    if key not in CACHE:
        Xtr = FEAT[tag][0]
        r = oof_for(Xtr, FOLDS[s], MAKER.get(tag, ridge_hid), SWS[sw])
        CACHE[key] = r if tag in TEXT_TAGS else (r + oof_for(Xtr, FOLDS[s], svr_head, SWS[sw])) / 2
    return CACHE[key]
def blend_oof(tags, s, sw='none'):
    return np.mean([comp_oof(t, s, sw) for t in tags], axis=0)
def proxies(tags, sw='none'):
    return {s: score(blend_oof(tags, s, sw))['proxy'] for s in ALL_S}

# ---------- integrity against notebooks 14 and 15 ----------
BASE4 = ['W', 'H', 'M7', 'M14']
EXP = {'base4 (v5 no-text)': (BASE4, {'A': 0.3440, 'B': 0.3643, 'C': 0.3378, 'g250': 0.3512, 'g120': 0.3540}),
       'v5 pick (with text)': (BASE4 + ['TL14'], {'A': 0.3466, 'B': 0.3650, 'C': 0.3442, 'g250': 0.3536, 'g120': 0.3548})}
for name, (tags, exp) in EXP.items():
    got = proxies(tags)
    tol = {s: (0.0015 if s in FRESH else 0.004) for s in ALL_S}
    bad = {s: (round(got[s], 4), exp[s]) for s in ALL_S if abs(got[s] - exp[s]) > tol[s]}
    print(f'integrity {name}:', {s: round(got[s], 4) for s in ALL_S}, '| OK' if not bad else f'| MISMATCH {bad}')

In [ ]:
def spec_feat(A, S, sp):
    if sp[0] == 'L':
        return A[:, sp[1]]
    if sp[0] == 'avg':
        return A[:, sp[1]:sp[2] + 1].mean(1)
    return np.hstack([A[:, sp[1]], S[:, sp[1]]])        # 'ms': mean + std of one layer

def sel_proxy(Xtr, maker):
    p = [score(oof_for(Xtr, FOLDS[s], maker))['proxy'] for s in SEL]
    return p[0], p[1], float(np.mean(p))

rows = []
for t, ((Mtr, Str), (Mte, Ste)) in ENC.items():
    specs = [('L', l) for l in range(2, 25, 2)] + [('avg', a, b) for a, b in [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24)]]
    for sp in specs:
        rows.append(dict(enc=t, spec=sp, **dict(zip(['g250', 'g120', 'mean'], sel_proxy(spec_feat(Mtr, Str, sp), ridge_hid)))))
    bestL = min((r for r in rows if r['enc'] == t and r['spec'][0] == 'L'), key=lambda r: r['mean'])['spec'][1]
    rows.append(dict(enc=t, spec=('ms', bestL), **dict(zip(['g250', 'g120', 'mean'], sel_proxy(spec_feat(Mtr, Str, ('ms', bestL)), ridge_hid)))))
SWP = pd.DataFrame(rows)
TAG = {'hubert': 'HB', 'w2v2': 'W2'}
for t, ((Mtr, Str), (Mte, Ste)) in ENC.items():
    sub = SWP[SWP.enc == t].sort_values('mean')
    print(f'\n{t}, ridge only, top 6 of {len(sub)} specs (selection folds):')
    print(sub.head(6).drop(columns='enc').round(4).to_string(index=False))
    sp = sub.iloc[0].spec
    FEAT[TAG[t]] = (spec_feat(Mtr, Str, sp), spec_feat(Mte, Ste, sp))
    print('chosen:', sp)
print('\nreference on the same folds, ridge only: whisper L30 0.3971/0.4025 | wavlm L21 0.4184/0.4309')

# existing W and H: switch to a layer band only if it beats the single layer by more than 0.003 on selection folds
REFINED = {}
for tag, A, maker, L, bands in [('W', AW, ridge_wavlm, 21, [(19, 23), (18, 24), (16, 24)]),
                                ('H', AH, ridge_hid, 30, [(28, 32), (26, 32), (24, 32)])]:
    single = sel_proxy(A[0][:, L], maker)[2]
    res = {b: sel_proxy(A[0][:, b[0]:b[1] + 1].mean(1), maker)[2] for b in bands}
    b = min(res, key=res.get)
    print(f'{tag}: single layer {L} {single:.4f} | bands', {k: round(v, 4) for k, v in res.items()})
    if res[b] < single - 0.003:
        REFINED[tag] = tag + 'r'
        FEAT[tag + 'r'] = (A[0][:, b[0]:b[1] + 1].mean(1), A[1][:, b[0]:b[1] + 1].mean(1))
        MAKER[tag + 'r'] = maker
        print(f'  -> candidate band {b}')
    else:
        print('  -> keep single layer')

In [ ]:
CANDS = {'C0 base4 (v5 no-text)': BASE4,
         'C1 + HuBERT': BASE4 + ['HB'],
         'C2 + wav2vec2': BASE4 + ['W2'],
         'C3 + HuBERT + wav2vec2': BASE4 + ['HB', 'W2'],
         'C4 + text (v5 pick)': BASE4 + ['TL14']}
if REFINED:
    CANDS['C5 refined W/H layers'] = [REFINED.get(t, t) for t in BASE4]
REF = 'C0 base4 (v5 no-text)'

def judge(pc, pr):
    d = np.array([pc[s] - pr[s] for s in ALL_S])
    fm, sm, wins = d[:3].mean(), d[3:].mean(), int((d[:3] < 0).sum())
    return d, fm, sm, wins, bool(wins == 3 and fm <= -TH and sm < 0)

P = {n: proxies(t) for n, t in CANDS.items()}
rows, PASS = [], {}
for n in CANDS:
    d, fm, sm, wins, ok = judge(P[n], P[REF])
    PASS[n] = ok and n != REF
    rows.append({'candidate': n, **{f'proxy_{s}': P[n][s] for s in ALL_S},
                 'fresh_mean': np.mean([P[n][s] for s in FRESH]),
                 'diff_fresh': fm, 'diff_sel': sm, 'fresh_wins': wins, 'PASS': PASS[n]})
DEC = pd.DataFrame(rows)
print(f'rule: 3/3 fresh wins AND fresh mean diff <= -{TH} AND selection-fold mean diff < 0 (diffs vs C0)\n')
print(DEC.round(4).to_string(index=False))

final = list(BASE4)
if PASS.get('C5 refined W/H layers'):
    final = [REFINED.get(t, t) for t in final]
for n, t in [('C1 + HuBERT', 'HB'), ('C2 + wav2vec2', 'W2'), ('C4 + text (v5 pick)', 'TL14')]:
    if PASS[n]:
        final.append(t)
if final != BASE4:
    Pf = proxies(final)
    d, fm, sm, wins, ok = judge(Pf, P[REF])
    print(f'\ncombined {final}: per scheme {np.round(d, 4)} | fresh mean {fm:+.4f} | wins {wins}/3 | PASS {ok}')
    if not ok:
        passed = [n for n in CANDS if PASS[n]]
        best = min(passed, key=lambda n: np.mean([P[n][s] for s in FRESH]))
        final = list(CANDS[best])
        print('combination failed the rule, falling back to the best single passing candidate:', best)
print('\nFINAL components:', final)
print('note: C3 is shown for information; an encoder joins only if it passes on its own (C1 or C2)')

In [ ]:
# Cell 3b: mean + std pooling for WavLM and Whisper, judged with the same rule against C0
EHS = np.load(find('whisper_enc_std.npy'), mmap_mode='r')
FEAT['Wms'] = (np.hstack([AW[0][:, 21], np.asarray(ES[w_tr, 21], dtype=np.float32)]),
               np.hstack([AW[1][:, 21], np.asarray(ES[w_te, 21], dtype=np.float32)]))
FEAT['Hms'] = (np.hstack([AH[0][:, 30], np.asarray(EHS[h_tr, 30], dtype=np.float32)]),
               np.hstack([AH[1][:, 30], np.asarray(EHS[h_te, 30], dtype=np.float32)]))
MAKER['Wms'] = ridge_wavlm
print('shapes:', FEAT['Wms'][0].shape, FEAT['Hms'][0].shape)

for t in ['W', 'Wms', 'H', 'Hms']:
    print(f'single mix model {t:4s}', {s: round(score(comp_oof(t, s))['proxy'], 4) for s in ALL_S})

MS = {'C6 WavLM mean+std': ['Wms', 'H', 'M7', 'M14'],
      'C7 Whisper mean+std': ['W', 'Hms', 'M7', 'M14'],
      'C8 both mean+std': ['Wms', 'Hms', 'M7', 'M14']}
rows = []
for n, tags in MS.items():
    P[n] = proxies(tags)
    d, fm, sm, wins, ok = judge(P[n], P[REF])
    PASS[n] = ok
    rows.append({'candidate': n, **{f'proxy_{s}': P[n][s] for s in ALL_S},
                 'diff_fresh': fm, 'diff_sel': sm, 'fresh_wins': wins, 'PASS': ok})
print(pd.DataFrame(rows).round(4).to_string(index=False))

# pick: C8 if it passes, otherwise whichever single swap passes with the better fresh mean
ms_pass = [n for n in MS if PASS[n]]
if ms_pass:
    best = 'C8 both mean+std' if PASS['C8 both mean+std'] else min(ms_pass, key=lambda n: np.mean([P[n][s] for s in FRESH]))
    swap = dict(zip(BASE4, MS[best]))
    final = [swap.get(t, t) for t in final]
    print('mean+std accepted:', best)
print('FINAL components after mean+std check:', final)

In [ ]:
# Cell 3c: 7B mean-pool on the literal CTC transcript, layer 16 fixed in advance (no layer search)
EC, c_tr, c_te = load_emb('llmc_meanpool.npy', 'llmc_uids.csv')
assert EC.shape == (985, 6, 3584)
FEAT['M7c'] = (np.asarray(EC[c_tr, j7], dtype=np.float32), np.asarray(EC[c_te, j7], dtype=np.float32))
for t in ['M7', 'M7c']:
    print(f'single mix model {t:4s}', {s: round(score(comp_oof(t, s))['proxy'], 4) for s in ALL_S})

P_ref = proxies(final)
CT = {'C9 + 7B on CTC transcript': final + ['M7c'],
      'C10 7B on CTC instead of Whisper transcript': ['M7c' if t == 'M7' else t for t in final]}
rows = []
for n, tags in CT.items():
    P[n] = proxies(tags)
    d, fm, sm, wins, ok = judge(P[n], P_ref)
    PASS[n] = ok
    rows.append({'candidate': n, **{f'proxy_{s}': P[n][s] for s in ALL_S},
                 'diff_fresh': fm, 'diff_sel': sm, 'fresh_wins': wins, 'PASS': ok})
print(pd.DataFrame(rows).round(4).to_string(index=False))
ct_pass = [n for n in CT if PASS[n]]
if ct_pass:
    best = min(ct_pass, key=lambda n: np.mean([P[n][s] for s in FRESH]))
    final = list(CT[best])
    print('CTC view accepted:', best)
print('FINAL components after CTC check:', final)

In [ ]:
SWS['sqrt'] = np.sqrt(w)           # half-strength version of the test-shift weights (45s 1.79, 60s 0.64)
P0, Pw = proxies(final), proxies(final, 'sqrt')
d, fm, sm, wins, ok = judge(Pw, P0)
print('sqrt(w_test) sample weights vs unweighted, per scheme', dict(zip(ALL_S, np.round(d, 4))))
print(f'fresh mean {fm:+.4f} | wins {wins}/3 | selection mean {sm:+.4f} | PASS {ok}')
SW_FINAL = 'sqrt' if ok else 'none'
print('weighting used in final model:', SW_FINAL)

In [ ]:
def calib_oof(o, fold):
    out, ab = np.full(n_tr, np.nan), []
    for f in range(5):
        va = fold == f
        tr = (~va) & nz
        a, b = np.polyfit(o[tr], y[tr], 1, w=np.sqrt(w[tr]))   # weighted least squares on the test weights
        ab.append((a, b))
        out[va & nz] = np.clip(a * o[va & nz] + b, 1, 5)
    return out, np.mean(ab, axis=0)

Pu, Pc, rows = proxies(final, SW_FINAL), {}, []
for s in ALL_S:
    o = blend_oof(final, s, SW_FINAL)
    c, (a, b) = calib_oof(o, FOLDS[s])
    su, sc_ = score(o), score(c)
    Pc[s] = sc_['proxy']
    rows.append(dict(scheme=s, slope=a, intercept=b, rmse_raw=su['rmse_w'], rmse_cal=sc_['rmse_w'],
                     pear_raw=su['pear_w'], pear_cal=sc_['pear_w'], proxy_raw=su['proxy'], proxy_cal=sc_['proxy']))
print(pd.DataFrame(rows).round(4).to_string(index=False))
d, fm, sm, wins, ok = judge(Pc, Pu)
print(f'calibration: fresh mean {fm:+.4f} | wins {wins}/3 | selection mean {sm:+.4f} | PASS {ok}')
CALIB = ok

In [ ]:
def fit_predict(tag, sw_name):
    Xtr, Xte = FEAT[tag]
    sw = SWS[sw_name]
    makers = [MAKER.get(tag, ridge_hid)] + ([] if tag in TEXT_TAGS else [svr_head])
    tr_p, te_p = [], []
    for mk in makers:
        m = fit_model(mk, Xtr[nz], y[nz], None if sw is None else sw[nz])
        tr_p.append(np.clip(m.predict(Xtr), 1, 5))
        te_p.append(np.clip(m.predict(Xte), 1, 5))
    return np.mean(tr_p, 0), np.mean(te_p, 0)

FIT = {t: fit_predict(t, SW_FINAL) for t in final}
REFP = {t: fit_predict(t, 'none') for t in set(BASE4) | {'TL14'}}
tr_in = np.mean([FIT[t][0] for t in final], axis=0)
pred = np.mean([FIT[t][1] for t in final], axis=0)

if CALIB:
    ab = []
    for s in ALL_S:
        o = blend_oof(final, s, SW_FINAL)
        ab.append(np.polyfit(o[nz], y[nz], 1, w=np.sqrt(w[nz])))
    a, b = np.mean(ab, axis=0)
    print(f'calibration applied to test: slope {a:.3f}, intercept {b:.3f} (mean over 5 schemes)')
    tr_in, pred = np.clip(a * tr_in + b, 1, 5), np.clip(a * pred + b, 1, 5)

# training RMSE (mandatory): in-sample, model fit on all non-zero train rows
print(f'\nTRAINING (in-sample, non-zero, n={nz.sum()}): RMSE {rmse(y[nz], tr_in[nz]):.4f} | Pearson {pear(y[nz], tr_in[nz]):.4f}')
if {'flat_mean', 'dyn_range'} <= set(AM.columns):
    rule = ((AM.flat_mean > 0.46) & (AM.dyn_range < 6)).reindex(folds.uid).values
    print(f'zero rule on train: flags {rule.sum()} files, of which {int((rule & ~nz).sum())} are true zeros (37 total)')
else:
    rule = ~nz
    print('acoustic columns not in audio_meta.csv; using the is_zero labels as the zero-rule output (rule caught all 37 in notebook 1)')
full = np.where(rule, 0.0, tr_in)
print(f'TRAINING (in-sample, all {n_tr} incl. zero rule): RMSE {rmse(y, full):.4f} | Pearson {pear(y, full):.4f}')

# out-of-fold numbers for the same final model
rows = []
for s in ALL_S:
    o = blend_oof(final, s, SW_FINAL)
    if CALIB:
        o = calib_oof(o, FOLDS[s])[0]
    r = score(o)
    rows.append(dict(scheme=s, oof_rmse=rmse(y[nz], o[nz]), oof_pearson=pear(y[nz], o[nz]), oof_std=float(np.std(o[nz])), **r))
OO = pd.DataFrame(rows)
print('\nOUT-OF-FOLD, final model:')
print(OO.round(4).to_string(index=False))
print(f'fresh-fold mean proxy {OO[OO.scheme.isin(FRESH)].proxy.mean():.4f} | selection-fold mean proxy {OO[OO.scheme.isin(SEL)].proxy.mean():.4f}')
print('the gap between training RMSE and OOF RMSE is expected: the SVR heads nearly memorise the training set')

# test sanity
p_c0 = np.mean([REFP[t][1] for t in BASE4], axis=0)
p_v5 = np.mean([REFP[t][1] for t in BASE4 + ['TL14']], axis=0)
print('\ntest pred by duration group:', pd.Series(pred).groupby(te_grp).agg(['count', 'mean', 'std']).round(3).to_dict('index'))
print('train labels by group (non-zero):', pd.Series(y[nz]).groupby(grp[nz]).agg(['count', 'mean', 'std']).round(3).to_dict('index'))
print(f'test pred std {np.std(pred):.3f} vs OOF pred std {OO.oof_std.mean():.3f} vs train label std {np.std(y[nz]):.3f}')
print(f'test corr with v5 no-text (LB 0.3700): {pear(pred, p_c0):.4f} | with v5 pick (LB 0.3662): {pear(pred, p_v5):.4f}')

# save
pd.DataFrame({'filename': test.filename, 'label': pred}).to_csv(f'{WORK}/submission_v6_final.csv', index=False)
oo = pd.DataFrame({'uid': folds.uid})
for (t, s, sw), v in CACHE.items():
    oo[f'{t}__{s}__{sw}'] = v
oo.to_csv(f'{WORK}/nb17_component_oof.csv', index=False)
pd.DataFrame({'filename': test.filename, **{t: FIT[t][1] for t in final}}).to_csv(f'{WORK}/nb17_component_test.csv', index=False)
DEC.to_csv(f'{WORK}/nb17_decisions.csv', index=False)
OO.to_csv(f'{WORK}/nb17_final_oof_scores.csv', index=False)
print('\nsaved submission_v6_final.csv, nb17_component_oof.csv, nb17_component_test.csv, nb17_decisions.csv, nb17_final_oof_scores.csv')
print('settings:', dict(final=final, weighting=SW_FINAL, calibration=CALIB, threshold=TH))

# plots
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
order = DEC.sort_values('fresh_mean', ascending=False)
ax[0].barh(order.candidate, order.fresh_mean, alpha=0.4)
for s in ALL_S:
    ax[0].scatter(order[f'proxy_{s}'], order.candidate, s=25, label=s, marker='o' if s in FRESH else 'x')
pcs = [f'proxy_{s}' for s in ALL_S]
ax[0].set_xlim(order[pcs].min().min() - 0.01, order[pcs].max().max() + 0.01)
ax[0].set_title('Proxy per scheme (circles fresh, crosses selection; lower is better)')
ax[0].legend()
o = blend_oof(final, 'A', SW_FINAL)
if CALIB:
    o = calib_oof(o, FOLDS['A'])[0]
sns.scatterplot(x=y[nz], y=o[nz], hue=grp[nz], alpha=0.5, ax=ax[1])
ax[1].plot([1, 5], [1, 5], 'r--'); ax[1].set_xlabel('true'); ax[1].set_ylabel('OOF prediction (scheme A)')
ax[1].set_title('Final model: predicted vs true')
plt.tight_layout(); plt.show()